# Conversational AI Chatbot with Gradio Interface

In [1]:
# Imports:

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [3]:
# Loading Environment Variables:

load_dotenv(override= True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print('OpenAI API KEY FOUND!')
else:
    print('OpenAI API KEY NOT FOUND!')

OpenAI API KEY FOUND!


In [15]:
# Initializing OpenAI LLM API:
openai = OpenAI()
model = 'gpt-4.1-mini'

In [5]:
# Initializing System Prompt:
system_message = 'You are a helpful assistant.'

## Writing a new callback:

We now need to write a function called:
`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.

In [6]:
def chat(message, history):
    return 'Shut Up!!'

In [8]:
gr.ChatInterface(fn= chat).launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [9]:
def chat(message, history):
    return f'You said {message} and the history is {history} but still, Shut Up!!'

In [11]:
gr.ChatInterface(fn= chat).launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Making a better callback function:

In [12]:
def chat(message, history):

    history = [{'role': h['role'], 'content': h['content']} for h in history] # Taking just the necessary parameters from history parameter

    messages = [{'role': 'system', 'content': system_message}] + history + [{'role': 'user', 'content': message}]

    response = openai.chat.completions.create(model= model, messages= messages)

    return response.choices[0].message.content

In [16]:
gr.ChatInterface(fn = chat).launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


## Making The Callback Function to Stream The Response:

In [20]:
def chat(message, history):

    history = [{'role': h['role'], 'content': h['content']} for h in history] # Taking just the necessary parameters from history parameter

    messages = [{'role': 'system', 'content': system_message}] + history + [{'role': 'user', 'content': message}]

    stream = openai.chat.completions.create(model= model, messages= messages, stream= True)

    response = ''

    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [21]:
gr.ChatInterface(fn = chat).launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


## Re-defining System prompt:

In [22]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [23]:
gr.ChatInterface(fn = chat).launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


In [24]:
# Adding more to System Prompt:

system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [25]:
gr.ChatInterface(fn = chat).launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


## Adding Relevant information to System Prompt based on Message:

In [28]:
def chat(message, history):

    history = [{'role': h['role'], 'content': h['content']} for h in history]

    relevant_system_message = system_message
    if 'belt' in message:
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."

    messages = [{'role': 'system', 'content': relevant_system_message}] + history + [{'role': 'user', 'content': message}]

    stream = openai.chat.completions.create(model= model, messages= messages, stream= True)

    response = ''
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [29]:
gr.ChatInterface(fn = chat).launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.
